In [12]:
import glob
import sqlite3
import pandas as pd

files = sorted(glob.glob("../data/raw/2025/*.csv"))
print(len(files))

def read_month(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        print("cp1252:", path)
        return pd.read_csv(path, encoding="cp1252")

df = pd.concat([read_month(f) for f in files], ignore_index=True)

12
cp1252: ../data/raw/2025/bikeshare_2025_01.csv
cp1252: ../data/raw/2025/bikeshare_2025_02.csv
cp1252: ../data/raw/2025/bikeshare_2025_03.csv
cp1252: ../data/raw/2025/bikeshare_2025_04.csv
cp1252: ../data/raw/2025/bikeshare_2025_05.csv
cp1252: ../data/raw/2025/bikeshare_2025_06.csv
cp1252: ../data/raw/2025/bikeshare_2025_07.csv
cp1252: ../data/raw/2025/bikeshare_2025_08.csv
cp1252: ../data/raw/2025/bikeshare_2025_09.csv
cp1252: ../data/raw/2025/bikeshare_2025_10.csv


Loading 2025 data

The 2025 ridership data is published as 12 monthly CSV files. I used glob to find all files in data/raw/2025/ and combined them into one DataFrame with pd.concat().

Encoding issue: read_csv() assumes UTF-8 by default, but loading failed with a UnicodeDecodeError. January to October 2025 were saved in the Windows cp1252 encoding, while November and December were UTF-8. The failing byte (0x96) is a dash character used in some station names. I wrote a small loader that tries UTF-8 first and falls back to cp1252, so each file is read with its correct encoding.

In [13]:
df.describe()

,Trip_Id,Trip_Duration,Start_Station_Id,End_Station_Id,Bike_Id
count,7.812520e+06,7.812520e+06,7.812513e+06,7.805759e+06,7.812520e+06
mean,3.919735e+07,9.722979e+02,7.418966e+03,7.414387e+03,5.523958e+03
std,2.614296e+06,1.464450e+03,3.256519e+02,3.253768e+02,3.091355e+03
min,3.463766e+07,0.000000e+00,7.000000e+03,7.000000e+03,1.400000e+01
25%,3.693459e+07,4.220000e+02,7.127000e+03,7.123000e+03,2.854000e+03
50%,3.922108e+07,6.810000e+02,7.342000e+03,7.333000e+03,5.591000e+03
75%,4.146559e+07,1.097000e+03,7.693000e+03,7.687000e+03,8.437000e+03
max,4.368241e+07,1.510370e+05,8.190000e+03,8.190000e+03,1.061800e+04


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7812520 entries, 0 to 7812519
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    float64
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(2), int64(3), str(6)
memory usage: 1.4 GB


In [15]:
df[["User_Type", "Bike_Model"]]

,User_Type,Bike_Model
0,Member,ICONIC
1,Member,ICONIC
2,Member,EFIT G5
3,Member,ICONIC
4,Member,ICONIC
...,...,...
7812515,Casual,ICONIC
7812516,Member,ICONIC
7812517,Member,ICONIC
7812518,Member,ICONIC


In [16]:
set(df["Bike_Model"])

{'EFIT', 'EFIT G5', 'ICONIC'}

In [17]:
df["is_ebike"] = df["Bike_Model"].isin(["EFIT","EFIT"])

In [18]:
df.columns

Index(['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time',
       'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name',
       'Bike_Id', 'User_Type', 'Bike_Model', 'is_ebike'],
      dtype='str')

In [41]:
df.isna().sum()

Trip_Id                  0
Trip_Duration            0
Start_Station_Id         0
Start_Time               0
Start_Station_Name       0
End_Station_Id        3596
End_Time                 0
End_Station_Name       343
Bike_Id                  0
User_Type                0
Bike_Model               0
is_ebike                 0
dtype: int64

In [20]:
df = df[df["Trip_Duration"] >= 60]

In [21]:
df["Trip_Duration"].quantile([0.99,0.999,0.9999])

0.9900     5524.0000
0.9990    14098.0000
0.9999    58325.6546
Name: Trip_Duration, dtype: float64

In [26]:
df = df[df["Trip_Duration"] <= 12875.00]

In [40]:
df.dtypes

Trip_Id                        int64
Trip_Duration                  int64
Start_Station_Id               Int64
Start_Time            datetime64[us]
Start_Station_Name               str
End_Station_Id                 Int64
End_Time              datetime64[us]
End_Station_Name                 str
Bike_Id                        int64
User_Type                        str
Bike_Model                       str
is_ebike                        bool
dtype: object

In [28]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"])
df["End_Time"] = pd.to_datetime(df["End_Time"])

In [31]:
df["End_Station_Id"] = df["End_Station_Id"].astype("Int64")
df["Start_Station_Id"] = df["Start_Station_Id"].astype("Int64")

In [33]:
(df["Trip_Id"].duplicated()).sum()

np.int64(0)

In [34]:
df["User_Type"].value_counts()

User_Type
Member    5653824
Casual    2129373
Name: count, dtype: int64

In [35]:
df["Bike_Model"].value_counts()

Bike_Model
ICONIC     6274249
EFIT       1020594
EFIT G5     488354
Name: count, dtype: int64

In [36]:
(df["Start_Station_Id"] == df["End_Station_Id"]).sum()

np.int64(242611)

In [37]:
round_trips = df[df["Start_Station_Id"] == df["End_Station_Id"]]

In [38]:
round_trips["Trip_Duration"].describe()

count    242611.000000
mean       2472.255083
std        2316.420950
min          61.000000
25%         627.000000
50%        1823.000000
75%        3692.000000
max       12874.000000
Name: Trip_Duration, dtype: float64

In [39]:
is_redock =(df["Start_Station_Id"] == df["End_Station_Id"]) & (df["Trip_Duration"] < 120)
df = df[~is_redock]

In [1]:
conn = sqlite3.connect("../data/processed/bikeshare.db")
df.to_sql("trips", conn, if_exists="append", index=False)
conn.close()

NameError: name 'sqlite3' is not defined